In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2
import zlib
import base64

In [2]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

In [3]:
medicina_general = pd.read_sql_query("SELECT * FROM progra_medicina_general", con=connection1) 
odontologia = pd.read_sql_query("SELECT * FROM progra_odontologia", con=connection1) 
pediatria = pd.read_sql_query("SELECT * FROM progra_pediatria", con=connection1)
ipress = pd.read_sql_query("SELECT * FROM ipress_ubicaciones", con=connection1)

In [4]:
# Crear un diccionario que mapee cada tipo de documento a su respectivo número
numeros_tipo_documento = {
    "D.N.I.": "1",
    "CARNE DE EXTRANJERIA/PASAPORTE": "2",
    "PASAPORTE": "3",
    "NEONATO": "8",
    "DOCUMENTO IDENTIDAD EXTRANJERO": "4",
    "PERMISO TEMPORAL DE PERMANENCIA": "5"
}

# Aplicar la transformación a la columna 'num_documento' utilizando el diccionario
medicina_general['num_documento'] = medicina_general['tipo_documento'].map(numeros_tipo_documento) + medicina_general['num_documento']
odontologia['num_documento'] = odontologia['tipo_documento'].map(numeros_tipo_documento) + odontologia['num_documento']
pediatria['num_documento'] = pediatria['tipo_documento'].map(numeros_tipo_documento) + pediatria['num_documento']

In [5]:
cas = pd.read_sql_query(f"SELECT cod_cas, des_cas FROM dim_cas where id_red is not null", con=connection1)

In [7]:
medicina_general = pd.merge(medicina_general,cas,how='left',left_on='ipress', right_on='des_cas')

In [8]:
medicina_general

,red,ipress,area_hospitalaria,servicio_hospitalario,actividad_hospitalaria,tipo_documento,num_documento,sum_properprohortot,properfec,propercrefec,pernacfec,perinsfec,cupos_totales,cod_cas,des_cas
0,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-02-27,2023-11-30 11:50:19,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN
1,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-02-26,2023-11-30 11:50:18,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN
2,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-02-05,2023-11-30 11:50:18,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN
3,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-01-08,2023-11-29 22:26:10,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN
4,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,2,2023-12-15,2023-12-15 10:47:11,1951-07-01 00:00:00,1984-04-05 00:00:00,8,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
721528,RED ASISTENCIAL PIURA,P.M. HUANCABAMBA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,174418039,4,2024-02-06,2023-10-11 12:49:36,1995-08-18 00:00:00,2023-10-24 00:00:00,16,NaN,NaN
721529,RED ASISTENCIAL PIURA,P.M. HUANCABAMBA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,174418039,4,2024-01-08,2023-10-11 12:47:36,1995-08-18 00:00:00,2023-10-24 00:00:00,16,NaN,NaN
721530,RED ASISTENCIAL MADRE DE DIOS,H.I VICTOR ALFREDO LAZO PERALTA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,129551558,4,2024-02-10,2024-01-12 11:57:50,1968-09-01 00:00:00,1998-09-21 00:00:00,16,NaN,NaN
721531,RED ASISTENCIAL MADRE DE DIOS,H.I VICTOR ALFREDO LAZO PERALTA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,129551558,4,2023-01-31,2022-12-31 09:53:23,1968-09-01 00:00:00,1998-09-21 00:00:00,16,NaN,NaN


In [122]:
archivo = '/home/ugadingenieria01/Documentos/GCTIC/DW_GCTIC/PNDA/UBIGEO 2022_1891 distritos.csv'
ubigeos = pd.read_csv(archivo, delimiter=';', header=0, encoding='utf-8')

In [123]:
ubigeos=ubigeos.rename(columns={"IDDIST":"UBIGEO"})
ubigeos=ubigeos.rename(columns={"NOMBDEP":"DEPARTAMENTO"})
ubigeos=ubigeos.rename(columns={"NOMBPROV":"PROVINCIA"})
ubigeos=ubigeos.rename(columns={"NOMBDIST":"DISTRITO"})

In [124]:
ipress = pd.merge(ipress,ubigeos,how='left',on=['DEPARTAMENTO','PROVINCIA','DISTRITO'])

In [126]:
ipress['COD EESS']=ipress['COD EESS'].astype('Int64').astype(str)

In [127]:
ipress=ipress.rename(columns={"COD EESS":"cod_cas"})

In [128]:
ipress = pd.merge(ipress,cas,how='left',on='cod_cas')

In [129]:
columnas_seleccionadas = ['DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'UBIGEO', 'des_cas']
ipress_final = ipress[columnas_seleccionadas].copy()

In [130]:
ipress_final['UBIGEO']=ipress_final['UBIGEO'].astype('Int64').astype(str)

In [131]:
ipress_final=ipress_final.rename(columns={"des_cas":"IPRESS"})

In [132]:
medicina_general.columns = medicina_general.columns.str.upper()
odontologia.columns = odontologia.columns.str.upper()
pediatria.columns = pediatria.columns.str.upper()

In [133]:
medicina_general=medicina_general.rename(columns={"PROPERFEC":"FEC_PROG_CUPO"})
medicina_general=medicina_general.rename(columns={"PROPERCREFEC":"FEC_CRE_PROG"})
medicina_general=medicina_general.rename(columns={"PERNACFEC":"FEC_NAC"})
medicina_general=medicina_general.rename(columns={"PERINSFEC":"FEC_INS_ESSALUD"})
odontologia=odontologia.rename(columns={"PROPERFEC":"FEC_PROG_CUPO"})
odontologia=odontologia.rename(columns={"PROPERCREFEC":"FEC_CRE_PROG"})
odontologia=odontologia.rename(columns={"PERNACFEC":"FEC_NAC"})
odontologia=odontologia.rename(columns={"PERINSFEC":"FEC_INS_ESSALUD"})
pediatria=pediatria.rename(columns={"PROPERFEC":"FEC_PROG_CUPO"})
pediatria=pediatria.rename(columns={"PROPERCREFEC":"FEC_CRE_PROG"})
pediatria=pediatria.rename(columns={"PERNACFEC":"FEC_NAC"})
pediatria=pediatria.rename(columns={"PERINSFEC":"FEC_INS_ESSALUD"})

In [134]:
medicina_general['IPRESS'] = medicina_general['IPRESS'].str.strip()

In [135]:
ipress_final['IPRESS'] = ipress_final['IPRESS'].str.strip()

In [136]:
medicina_general= pd.merge(medicina_general,ipress_final,how='left',on='IPRESS')

In [142]:
ipress_final

,DEPARTAMENTO,PROVINCIA,DISTRITO,UBIGEO,IPRESS
0,AMAZONAS,UTCUBAMBA,BAGUA GRANDE,10701,H.I EL BUEN SAMARITANO
1,AMAZONAS,BAGUA,LA PECA,10206,H.I HEROES DEL CENEPA
2,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,10101,H.I HIGOS URCO
3,AMAZONAS,RODRÍGUEZ DE MENDOZA,SAN NICOLÁS,<NA>,CAP II RODRIGUEZ DE MENDOZA
4,AMAZONAS,CONDORCANQUI,NIEVA,10401,CAP II SANTA MARIA DE NIEVA
...,...,...,...,...,...
395,LIMA,LIMA,VILLA MARÍA DEL TRIUNFO,<NA>,NaN
396,AREQUIPA,AREQUIPA,CERRO COLORADO,40104,NaN
397,PIURA,PIURA,CASTILLA,200104,NaN
398,PIURA,SULLANA,SULLANA,200601,NaN


In [152]:
ipress_final_cleaned = ipress_final.dropna(subset=['IPRESS'])

In [156]:
resultados = ipress_final_cleaned[ipress_final_cleaned['IPRESS'].str.contains('RAMON CASTILLA')]

In [157]:
resultados

,DEPARTAMENTO,PROVINCIA,DISTRITO,UBIGEO,IPRESS


In [137]:
medicina_general

,RED,IPRESS,AREA_HOSPITALARIA,SERVICIO_HOSPITALARIO,ACTIVIDAD_HOSPITALARIA,TIPO_DOCUMENTO,NUM_DOCUMENTO,SUM_PROPERPROHORTOT,FEC_PROG_CUPO,FEC_CRE_PROG,FEC_NAC,FEC_INS_ESSALUD,CUPOS_TOTALES,DEPARTAMENTO,PROVINCIA,DISTRITO,UBIGEO
0,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-02-27,2023-11-30 11:50:19,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN,NaN,NaN
1,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-02-26,2023-11-30 11:50:18,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN,NaN,NaN
2,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-02-05,2023-11-30 11:50:18,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN,NaN,NaN
3,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,3,2024-01-08,2023-11-29 22:26:10,1951-07-01 00:00:00,1984-04-05 00:00:00,12,NaN,NaN,NaN,NaN
4,RED ASISTENCIAL ALMENARA,H.II RAMON CASTILLA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,107958782,2,2023-12-15,2023-12-15 10:47:11,1951-07-01 00:00:00,1984-04-05 00:00:00,8,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
721528,RED ASISTENCIAL PIURA,P.M. HUANCABAMBA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,174418039,4,2024-02-06,2023-10-11 12:49:36,1995-08-18 00:00:00,2023-10-24 00:00:00,16,PIURA,HUANCABAMBA,HUANCABAMBA,200301
721529,RED ASISTENCIAL PIURA,P.M. HUANCABAMBA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,174418039,4,2024-01-08,2023-10-11 12:47:36,1995-08-18 00:00:00,2023-10-24 00:00:00,16,PIURA,HUANCABAMBA,HUANCABAMBA,200301
721530,RED ASISTENCIAL MADRE DE DIOS,H.I VICTOR ALFREDO LAZO PERALTA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,129551558,4,2024-02-10,2024-01-12 11:57:50,1968-09-01 00:00:00,1998-09-21 00:00:00,16,MADRE DE DIOS,TAMBOPATA,TAMBOPATA,170101
721531,RED ASISTENCIAL MADRE DE DIOS,H.I VICTOR ALFREDO LAZO PERALTA,CONSULTA EXTERNA,MEDICINA GENERAL,ATENCION MEDICA AMBULATORIA,D.N.I.,129551558,4,2023-01-31,2022-12-31 09:53:23,1968-09-01 00:00:00,1998-09-21 00:00:00,16,MADRE DE DIOS,TAMBOPATA,TAMBOPATA,170101


In [138]:
medicina_general_filtrado = medicina_general.loc[medicina_general['DEPARTAMENTO'].isna()]

In [140]:
ipress_unicos = medicina_general_filtrado['IPRESS'].unique()

In [141]:
ipress_unicos

array(['H.II RAMON CASTILLA', 'CENTRO SALUD ES HOSPITAL UNIVERSITARIO',
       'POL. CHINCHA', 'H.II LIMA NORTE - CALLAO L. NEGREIROS V.',
       'C.M. ANCIJE', 'PADOMI', 'POL. COMPLEJIDAD CRECIENTE SAN NICOLAS',
       'POL. PABLO BERMUDEZ', 'H.II RENE TOCHE GROPPO',
       'CENT.DE COMPLEJ.CRECIENTE CERRO COLORADO', 'H.III YANAHUARA',
       'IPRESS CLINICA INMACULADA', 'POLICLINICO BAYOVAR',
       'HOSPITAL PRIVADO DEL PERU', 'P.M. CHUQUIBAMBILLA', 'C.M. MALA',
       'H.II CAÑETE', 'POL. PROCERES', 'C.M. CASAPALCA',
       'IPRESS SAN MARTIN DE PORRES', 'POL. FRANCISCO PIZARRO',
       'POL. FIORI', 'H.III HOSPITAL DE EMERGENCIAS GRAU', 'H.II VITARTE',
       'H.I YURIMAGUAS', 'H.I ANTONIO SKRABONJA ANTONCICH',
       'CAP II CHALHUANCA', 'H.I SAMUEL PASTOR', 'POL. CHOSICA',
       'CAP III HUARAL', 'CAP III PEDRO REYES BARBOZA', 'P.M. LA JOYA',
       'H.II MANUEL DE TORRES MUÑOZ', 'CAP I EL PEDREGAL',
       'POL. METROPOLITANO AREQUIPA', 'CAP II HUNTER', 'CAP II NAUTA',
       

In [5]:
print(medicina_general.shape)
medicina_general['FEC_NAC'] = pd.to_datetime(medicina_general['FEC_NAC'], errors='coerce')
medicina_general['FEC_INS_ESSALUD'] = pd.to_datetime(medicina_general['FEC_INS_ESSALUD'], errors='coerce')
medicina_general = medicina_general[medicina_general['FEC_NAC'].notnull()]
medicina_general = medicina_general[medicina_general['FEC_NAC'].notna()]
medicina_general = medicina_general[medicina_general['FEC_INS_ESSALUD'].notnull()]
medicina_general = medicina_general[medicina_general['FEC_INS_ESSALUD'].notna()]
# Convertir la columna original a bytes
medicina_general['ID_PERSONAL'] = (medicina_general['NUM_DOCUMENTO'].astype(int)*medicina_general['NUM_DOCUMENTO'].astype(int)).astype(str).apply(lambda x: x.encode())
# Comprimir la columna original
medicina_general['ID_PERSONAL'] = medicina_general['ID_PERSONAL'].apply(lambda x: zlib.compress(x))
# Convertir la columna comprimida a base64 y luego a un string
medicina_general['ID_PERSONAL'] = medicina_general['ID_PERSONAL'].apply(lambda x: base64.b64encode(x).decode())
medicina_general.drop('TIPO_DOCUMENTO', axis=1)
medicina_general.drop('NUM_DOCUMENTO', axis=1)
medicina_general['DIFERIMIENTO_PROGRAMACION'] = (medicina_general['FEC_PROG_CUPO'].dt.date - medicina_general['FEC_CRE_PROG'].dt.date).dt.days
medicina_general = medicina_general[medicina_general['DIFERIMIENTO_PROGRAMACION']>=0]
medicina_general=medicina_general.drop('TIPO_DOCUMENTO',axis=1)
medicina_general=medicina_general.drop('NUM_DOCUMENTO',axis=1)
medicina_general=medicina_general.rename(columns={'SUM_PROPERPROHORTOT':'SUMA_HORAS_PROG'})
print(medicina_general.shape)
medicina_general.to_csv('DataSet_ProgramacionAsistencial_MedicinaGeneral_10.2022_03.2024.csv', index=False, sep=';')

(721533, 13)
(716642, 13)


In [17]:
print(odontologia.shape)
odontologia['pernacfec'] = pd.to_datetime(odontologia['pernacfec'], errors='coerce')
odontologia['perinsfec'] = pd.to_datetime(odontologia['perinsfec'], errors='coerce')
odontologia = odontologia[odontologia['pernacfec'].notnull()]
odontologia = odontologia[odontologia['pernacfec'].notna()]
odontologia = odontologia[odontologia['perinsfec'].notnull()]
odontologia = odontologia[odontologia['perinsfec'].notna()]
# Convertir la columna original a bytes
odontologia['id_personal'] = (odontologia['num_documento'].astype(int)*odontologia['num_documento'].astype(int)).astype(str).apply(lambda x: x.encode())
# Comprimir la columna original
odontologia['id_personal'] = odontologia['id_personal'].apply(lambda x: zlib.compress(x))
# Convertir la columna comprimida a base64 y luego a un string
odontologia['id_personal'] = odontologia['id_personal'].apply(lambda x: base64.b64encode(x).decode())
odontologia.drop('tipo_documento', axis=1)
odontologia.drop('num_documento', axis=1)
odontologia['diferimiento_programacion'] = (odontologia['properfec'].dt.date - odontologia['propercrefec'].dt.date).dt.days
odontologia = odontologia[odontologia['diferimiento_programacion']>=0]
odontologia=odontologia.drop('tipo_documento',axis=1)
odontologia=odontologia.drop('num_documento',axis=1)
odontologia=odontologia.rename(columns={'pernacfec':'fec_nac'})
odontologia=odontologia.rename(columns={'perinsfec':'fec_ins_essalud'})
odontologia=odontologia.rename(columns={'propercrefec':'fec_cre_prog'})
odontologia=odontologia.rename(columns={'properfec':'fec_prog_cupo'})
odontologia=odontologia.rename(columns={'sum_properprohortot':'suma_horas_prog'})
print(odontologia.shape)
odontologia.to_csv('DataSet_ProgramacionAsistencial_Odontologia_10.2022_03.2024.csv', index=False, sep=';')

(367433, 13)
(364892, 13)


In [18]:
print(pediatria.shape)
pediatria['pernacfec'] = pd.to_datetime(pediatria['pernacfec'], errors='coerce')
pediatria['perinsfec'] = pd.to_datetime(pediatria['perinsfec'], errors='coerce')
pediatria = pediatria[pediatria['pernacfec'].notnull()]
pediatria = pediatria[pediatria['pernacfec'].notna()]
pediatria = pediatria[pediatria['perinsfec'].notnull()]
pediatria = pediatria[pediatria['perinsfec'].notna()]
# Convertir la columna original a bytes
pediatria['id_personal'] = (pediatria['num_documento'].astype(int)*pediatria['num_documento'].astype(int)).astype(str).apply(lambda x: x.encode())
# Comprimir la columna original
pediatria['id_personal'] = pediatria['id_personal'].apply(lambda x: zlib.compress(x))
# Convertir la columna comprimida a base64 y luego a un string
pediatria['id_personal'] = pediatria['id_personal'].apply(lambda x: base64.b64encode(x).decode())
pediatria.drop('tipo_documento', axis=1)
pediatria.drop('num_documento', axis=1)
pediatria['diferimiento_programacion'] = (pediatria['properfec'].dt.date - pediatria['propercrefec'].dt.date).dt.days
pediatria = pediatria[pediatria['diferimiento_programacion']>=0]
pediatria=pediatria.drop('tipo_documento',axis=1)
pediatria=pediatria.drop('num_documento',axis=1)
pediatria=pediatria.rename(columns={'pernacfec':'fec_nac'})
pediatria=pediatria.rename(columns={'perinsfec':'fec_ins_essalud'})
pediatria=pediatria.rename(columns={'propercrefec':'fec_cre_prog'})
pediatria=pediatria.rename(columns={'properfec':'fec_prog_cupo'})
pediatria=pediatria.rename(columns={'sum_properprohortot':'suma_horas_prog'})
print(pediatria.shape)
pediatria.to_csv('DataSet_ProgramacionAsistencial_Pediatria_10.2022_03.2024.csv', index=False, sep=';')

(129621, 13)
(129230, 13)


In [10]:
pediatria.shape

(129230, 13)

In [11]:
odontologia.shape

(364892, 13)

In [12]:
medicina_general.shape

(716642, 13)